# AIF360 Fairness Audit — Calculation Notebook

This notebook computes post-hoc subgroup fairness metrics for ICU mortality prediction using the existing clinical preprocessing pipeline. It does **not** apply AIF360 fairness-mitigation preprocessing algorithms such as reweighing or optimized preprocessing.

Expected project layout when this notebook is placed inside the `code/` folder:

```text
AIF360/
├── code/
│   └── 01_aif360_fairness_calculation.ipynb
└── data/
    ├── X_train.csv
    ├── X_test.csv
    ├── y_train.csv
    ├── y_test.csv
    └── cohort_distribution_dataset.csv
```


In [7]:
#Optional installation
# Run this only if AIF360 is not installed in your environment.
#!pip install aif360 pandas numpy scikit-learn matplotlib xgboost

In [8]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)
RANDOM_STATE = 42

In [9]:
NOTEBOOK_DIR = Path.cwd()

# If the notebook is executed from AIF360/code, the project root is AIF360.
# If it is executed from AIF360, the project root is the current directory.
if NOTEBOOK_DIR.name.lower() == "code":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "aif360"
FIGURES_DIR = PROJECT_ROOT / "figures" / "aif360"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Data directory:    {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Notebook directory: c:\Users\junse\Documents\research\IUBDC 2026\AIF360\code
Project root:       c:\Users\junse\Documents\research\IUBDC 2026\AIF360
Data directory:    c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data
Results directory: c:\Users\junse\Documents\research\IUBDC 2026\AIF360\results\aif360


In [10]:
def resolve_input_file(canonical_name, glob_patterns):
    # Resolve a required input file from the local data folder.
    # The canonical names match the intended project structure.
    # The glob patterns also allow uploaded copies such as X_train(9).csv.
    search_dirs = [DATA_DIR, NOTEBOOK_DIR, PROJECT_ROOT, Path("/mnt/data")]
    checked = []

    for directory in search_dirs:
        candidate = directory / canonical_name
        checked.append(candidate)
        if candidate.exists():
            return candidate

    for directory in search_dirs:
        for pattern in glob_patterns:
            matches = sorted(directory.glob(pattern))
            if matches:
                return matches[0]

    checked_text = "\n".join(str(p) for p in checked)
    raise FileNotFoundError(
        f"Could not find {canonical_name}. Checked canonical paths:\n{checked_text}"
    )

x_train_path = resolve_input_file("X_train.csv", ["X_train*.csv"])
x_test_path = resolve_input_file("X_test.csv", ["X_test*.csv"])
y_train_path = resolve_input_file("y_train.csv", ["y_train*.csv"])
y_test_path = resolve_input_file("y_test.csv", ["y_test*.csv"])
cohort_path = resolve_input_file("cohort_distribution_dataset.csv", ["cohort_distribution_dataset*.csv"])

print("Resolved input files:")
for path in [x_train_path, x_test_path, y_train_path, y_test_path, cohort_path]:
    print(" -", path)

Resolved input files:
 - c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data\X_train.csv
 - c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data\X_test.csv
 - c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data\y_train.csv
 - c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data\y_test.csv
 - c:\Users\junse\Documents\research\IUBDC 2026\AIF360\data\cohort_distribution_dataset.csv


In [11]:
#Load data
X_train = pd.read_csv(x_train_path)
X_test = pd.read_csv(x_test_path)
y_train_df = pd.read_csv(y_train_path)
y_test_df = pd.read_csv(y_test_path)
cohort = pd.read_csv(cohort_path)

TARGET_COL = "in_hospital_death" if "in_hospital_death" in y_train_df.columns else y_train_df.columns[0]
y_train = y_train_df[TARGET_COL].astype(int).reset_index(drop=True)
y_test = y_test_df[TARGET_COL].astype(int).reset_index(drop=True)

cohort_train = cohort.loc[cohort["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
cohort_test = cohort.loc[cohort["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

assert len(X_train) == len(y_train) == len(cohort_train), "Train files are not aligned by row count."
assert len(X_test) == len(y_test) == len(cohort_test), "Test files are not aligned by row count."

if TARGET_COL in cohort_train.columns:
    assert np.array_equal(y_train.values, cohort_train[TARGET_COL].astype(int).values), "Train labels do not match cohort metadata."
if TARGET_COL in cohort_test.columns:
    assert np.array_equal(y_test.values, cohort_test[TARGET_COL].astype(int).values), "Test labels do not match cohort metadata."

print("Loaded data successfully.")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Train mortality rate: {y_train.mean():.3f}")
print(f"Test mortality rate:  {y_test.mean():.3f}")

Loaded data successfully.
X_train: (8000, 243)
X_test:  (4000, 243)
Train mortality rate: 0.140
Test mortality rate:  0.146


In [12]:
#Import AIF360 metrics when available
try:
    from aif360.sklearn.metrics import (
        statistical_parity_difference as aif360_statistical_parity_difference,
        disparate_impact_ratio as aif360_disparate_impact_ratio,
        equal_opportunity_difference as aif360_equal_opportunity_difference,
        average_odds_difference as aif360_average_odds_difference,
    )
    AIF360_AVAILABLE = True
except Exception as exc:
    AIF360_AVAILABLE = False
    AIF360_IMPORT_ERROR = str(exc)

print(f"AIF360 available: {AIF360_AVAILABLE}")
if not AIF360_AVAILABLE:
    print("AIF360 was not available. The notebook will use formula-equivalent fallback metrics.")
    print("Install with: pip install aif360")

AIF360 available: False
AIF360 was not available. The notebook will use formula-equivalent fallback metrics.
Install with: pip install aif360


In [13]:
#Helper functions
def safe_divide(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan
    return numerator / denominator


def binary_confusion_rates(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    if len(np.unique(y_true)) < 2:
        tn = fp = fn = tp = 0
        for yt, yp in zip(y_true, y_pred):
            if yt == 0 and yp == 0:
                tn += 1
            elif yt == 0 and yp == 1:
                fp += 1
            elif yt == 1 and yp == 0:
                fn += 1
            elif yt == 1 and yp == 1:
                tp += 1
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tpr": safe_divide(tp, tp + fn),
        "fpr": safe_divide(fp, fp + tn),
        "fnr": safe_divide(fn, fn + tp),
        "tnr": safe_divide(tn, tn + fp),
        "ppv": safe_divide(tp, tp + fp),
        "npv": safe_divide(tn, tn + fn),
    }


def model_performance_row(model_name, y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "model": model_name,
        "n": len(y_true),
        "threshold": threshold,
        "mortality_rate": float(np.mean(y_true)),
        "predicted_positive_rate": float(np.mean(y_pred)),
        "mean_predicted_risk": float(np.mean(y_prob)),
        "auroc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "auprc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def subgroup_performance_rows(model_name, y_true, y_prob, subgroup_df, threshold=0.5):
    rows = []
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_prob = pd.Series(y_prob).reset_index(drop=True)

    for _, row in subgroup_df.iterrows():
        attribute = row["attribute"]
        group = row["group"]
        mask = row["mask"]
        yt = y_true[mask]
        yp = y_prob[mask]

        perf = model_performance_row(model_name, yt, yp, threshold)
        perf.update({
            "attribute": attribute,
            "group": group,
            "comparison": row.get("comparison", group),
        })
        rows.append(perf)

    return pd.DataFrame(rows)


def formula_fairness_metrics(y_true, y_pred, protected_attribute, privileged_value):
    y_true = pd.Series(y_true).reset_index(drop=True).astype(int)
    y_pred = pd.Series(y_pred).reset_index(drop=True).astype(int)
    protected_attribute = pd.Series(protected_attribute).reset_index(drop=True)

    privileged_mask = protected_attribute == privileged_value
    unprivileged_mask = ~privileged_mask

    y_true_priv = y_true[privileged_mask]
    y_pred_priv = y_pred[privileged_mask]
    y_true_unpriv = y_true[unprivileged_mask]
    y_pred_unpriv = y_pred[unprivileged_mask]

    selection_priv = float(y_pred_priv.mean()) if len(y_pred_priv) else np.nan
    selection_unpriv = float(y_pred_unpriv.mean()) if len(y_pred_unpriv) else np.nan

    rates_priv = binary_confusion_rates(y_true_priv, y_pred_priv)
    rates_unpriv = binary_confusion_rates(y_true_unpriv, y_pred_unpriv)

    spd = selection_unpriv - selection_priv
    di = safe_divide(selection_unpriv, selection_priv)
    eod = rates_unpriv["tpr"] - rates_priv["tpr"]
    aod = 0.5 * ((rates_unpriv["fpr"] - rates_priv["fpr"]) + (rates_unpriv["tpr"] - rates_priv["tpr"]))

    return {
        "statistical_parity_difference": spd,
        "disparate_impact_ratio": di,
        "equal_opportunity_difference": eod,
        "average_odds_difference": aod,
        "selection_rate_privileged": selection_priv,
        "selection_rate_unprivileged": selection_unpriv,
        "tpr_privileged": rates_priv["tpr"],
        "tpr_unprivileged": rates_unpriv["tpr"],
        "fpr_privileged": rates_priv["fpr"],
        "fpr_unprivileged": rates_unpriv["fpr"],
    }


def compute_fairness_metrics(y_true, y_pred, protected_attribute, privileged_value):
    # Compute AIF360 fairness metrics with a formula-equivalent fallback.
    fallback = formula_fairness_metrics(y_true, y_pred, protected_attribute, privileged_value)

    if not AIF360_AVAILABLE:
        fallback["metric_backend"] = "formula_equivalent_to_aif360_definitions"
        return fallback

    try:
        y_true_s = pd.Series(y_true).reset_index(drop=True).astype(int)
        y_pred_s = pd.Series(y_pred).reset_index(drop=True).astype(int)
        prot_s = pd.Series(protected_attribute).reset_index(drop=True)

        fallback["statistical_parity_difference"] = aif360_statistical_parity_difference(
            y_true_s, y_pred_s, prot_attr=prot_s, priv_group=privileged_value, pos_label=1
        )
        fallback["disparate_impact_ratio"] = aif360_disparate_impact_ratio(
            y_true_s, y_pred_s, prot_attr=prot_s, priv_group=privileged_value, pos_label=1
        )
        fallback["equal_opportunity_difference"] = aif360_equal_opportunity_difference(
            y_true_s, y_pred_s, prot_attr=prot_s, priv_group=privileged_value, pos_label=1
        )
        fallback["average_odds_difference"] = aif360_average_odds_difference(
            y_true_s, y_pred_s, prot_attr=prot_s, priv_group=privileged_value, pos_label=1
        )
        fallback["metric_backend"] = "aif360.sklearn.metrics"
    except Exception as exc:
        fallback["metric_backend"] = f"formula_fallback_after_aif360_error: {exc}"

    return fallback

In [14]:
#Train baseline models and generate external-test predictions
# The notebook trains a compact XGBoost model when xgboost is available.
# Random Forest is kept as a fast fallback/comparator. If you already have final
# model predictions, replace this block with a CSV load and keep the same output
# columns: row_id, model, y_true, y_prob, y_pred.
models = {}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - y_train.sum()) / max(y_train.sum(), 1),
        random_state=RANDOM_STATE,
        n_jobs=4,
    )
except Exception as exc:
    print(f"XGBoost is not available and will be skipped: {exc}")

models["Random Forest"] = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

prediction_frames = []
performance_rows = []
for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    prediction_frames.append(pd.DataFrame({
        "row_id": np.arange(len(y_test)),
        "model": model_name,
        "y_true": y_test.values,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }))
    performance_rows.append(model_performance_row(model_name, y_test, y_prob, threshold=0.5))

predictions = pd.concat(prediction_frames, ignore_index=True)
model_performance = pd.DataFrame(performance_rows).sort_values("auprc", ascending=False)

preferred_order = ["XGBoost", "Random Forest"]
available_models = set(predictions["model"].unique())
PRIMARY_MODEL = next(model for model in preferred_order if model in available_models)

predictions.to_csv(RESULTS_DIR / "prediction_outputs_external_test.csv", index=False)
model_performance.to_csv(RESULTS_DIR / "model_performance_external_test.csv", index=False)
(RESULTS_DIR / "primary_model_name.txt").write_text(PRIMARY_MODEL)

print("Primary model for fairness figures:", PRIMARY_MODEL)
display(model_performance)

Training XGBoost...
Training Random Forest...
Primary model for fairness figures: XGBoost


,model,n,threshold,mortality_rate,predicted_positive_rate,mean_predicted_risk,auroc,auprc,brier,accuracy,precision,recall,f1
0,XGBoost,4000,0.5,0.14625,0.284,0.344430,0.866808,0.549991,0.139765,0.79125,0.389965,0.757265,0.514817
1,Random Forest,4000,0.5,0.14625,0.142,0.257778,0.854814,0.508678,0.108514,0.86125,0.526408,0.511111,0.518647


In [15]:
#Define subgroup variables for fairness auditing
cohort_test_for_audit = cohort_test.copy().reset_index(drop=True)
cohort_test_for_audit["Age_group"] = np.where(cohort_test_for_audit["Age"] > 65, ">65", "<=65")
cohort_test_for_audit["Gender_group"] = cohort_test_for_audit["Gender_label"].astype(str)
cohort_test_for_audit["ICUType_group"] = cohort_test_for_audit["ICUType_label"].astype(str)

subgroup_rows = []
for group in ["<=65", ">65"]:
    subgroup_rows.append({"attribute": "Age", "group": group, "comparison": group, "mask": (cohort_test_for_audit["Age_group"] == group).values})
for group in ["Female", "Male"]:
    subgroup_rows.append({"attribute": "Gender", "group": group, "comparison": group, "mask": (cohort_test_for_audit["Gender_group"] == group).values})
for group in sorted(cohort_test_for_audit["ICUType_group"].dropna().unique()):
    subgroup_rows.append({"attribute": "ICUType", "group": group, "comparison": group, "mask": (cohort_test_for_audit["ICUType_group"] == group).values})
subgroup_df = pd.DataFrame(subgroup_rows)

composition_rows = []
for _, row in subgroup_df.iterrows():
    mask = row["mask"]
    composition_rows.append({
        "attribute": row["attribute"],
        "group": row["group"],
        "comparison": row["comparison"],
        "n": int(mask.sum()),
        "mortality_rate": float(y_test[mask].mean()) if mask.sum() else np.nan,
    })
subgroup_composition = pd.DataFrame(composition_rows)
subgroup_composition.to_csv(RESULTS_DIR / "cohort_subgroup_composition.csv", index=False)
display(subgroup_composition)

,attribute,group,comparison,n,mortality_rate
0,Age,<=65,<=65,1894,0.109293
1,Age,>65,>65,2106,0.179487
2,Gender,Female,Female,1768,0.161199
3,Gender,Male,Male,2228,0.134650
4,ICUType,Cardiac Surgery Recovery Unit,Cardiac Surgery Recovery Unit,874,0.052632
5,ICUType,Coronary Care Unit,Coronary Care Unit,602,0.117940
6,ICUType,Medical ICU,Medical ICU,1376,0.211483
7,ICUType,Surgical ICU,Surgical ICU,1148,0.154181


In [16]:
# Compute subgroup performance for the primary model
primary_pred = predictions.loc[predictions["model"] == PRIMARY_MODEL].sort_values("row_id").reset_index(drop=True)
y_prob_primary = primary_pred["y_prob"].values
y_pred_primary = primary_pred["y_pred"].values

subgroup_performance = subgroup_performance_rows(PRIMARY_MODEL, y_test, y_prob_primary, subgroup_df, threshold=0.5)
subgroup_performance.to_csv(RESULTS_DIR / "subgroup_performance_primary_model.csv", index=False)
display(subgroup_performance)

,model,n,threshold,mortality_rate,predicted_positive_rate,mean_predicted_risk,auroc,auprc,brier,accuracy,precision,recall,f1,attribute,group,comparison
0,XGBoost,1894,0.5,0.109293,0.197994,0.282408,0.878090,0.520000,0.109734,0.846885,0.389333,0.705314,0.501718,Age,<=65,<=65
1,XGBoost,2106,0.5,0.179487,0.361349,0.400209,0.849534,0.568873,0.166772,0.741216,0.390276,0.785714,0.521510,Age,>65,>65
2,XGBoost,1768,0.5,0.161199,0.303733,0.369167,0.859971,0.557900,0.150631,0.780543,0.404097,0.761404,0.527981,Gender,Female,Female
3,XGBoost,2228,0.5,0.134650,0.268402,0.324873,0.871774,0.545425,0.131175,0.799820,0.377926,0.753333,0.503341,Gender,Male,Male
4,XGBoost,874,0.5,0.052632,0.091533,0.172239,0.895636,0.488108,0.060197,0.924485,0.375000,0.652174,0.476190,ICUType,Cardiac Surgery Recovery Unit,Cardiac Surgery Recovery Unit
5,XGBoost,602,0.5,0.117940,0.235880,0.334826,0.828890,0.450109,0.142327,0.785714,0.295775,0.591549,0.394366,ICUType,Coronary Care Unit,Coronary Care Unit
6,XGBoost,1376,0.5,0.211483,0.425872,0.446783,0.832777,0.596945,0.186328,0.707122,0.404437,0.814433,0.540479,ICUType,Medical ICU,Medical ICU
7,XGBoost,1148,0.5,0.154181,0.285714,0.357879,0.866635,0.527811,0.143187,0.793554,0.408537,0.757062,0.530693,ICUType,Surgical ICU,Surgical ICU


In [17]:
# Compute AIF360-style fairness comparisons for the primary model
fairness_specs = []
fairness_specs.append({
    "attribute": "Age",
    "comparison": ">65 vs <=65",
    "protected_attribute": cohort_test_for_audit["Age_group"],
    "privileged_value": "<=65",
    "included_mask": np.ones(len(cohort_test_for_audit), dtype=bool),
})

gender_mask = cohort_test_for_audit["Gender_group"].isin(["Female", "Male"]).values
fairness_specs.append({
    "attribute": "Gender",
    "comparison": "Female vs Male",
    "protected_attribute": cohort_test_for_audit.loc[gender_mask, "Gender_group"].reset_index(drop=True),
    "privileged_value": "Male",
    "included_mask": gender_mask,
})

for icu_group in sorted(cohort_test_for_audit["ICUType_group"].dropna().unique()):
    prot = pd.Series(np.where(cohort_test_for_audit["ICUType_group"] == icu_group, icu_group, "Other"))
    fairness_specs.append({
        "attribute": "ICUType",
        "comparison": f"Other vs {icu_group}",
        "protected_attribute": prot,
        "privileged_value": icu_group,
        "included_mask": np.ones(len(cohort_test_for_audit), dtype=bool),
    })

fairness_rows = []
confusion_rows = []
observed_predicted_rows = []
for spec in fairness_specs:
    mask = spec["included_mask"]
    yt = pd.Series(y_test[mask]).reset_index(drop=True)
    yh = pd.Series(y_pred_primary[mask]).reset_index(drop=True)
    yp = pd.Series(y_prob_primary[mask]).reset_index(drop=True)
    prot = pd.Series(spec["protected_attribute"]).reset_index(drop=True)
    priv = spec["privileged_value"]
    metrics = compute_fairness_metrics(yt, yh, prot, priv)
    metrics.update({
        "model": PRIMARY_MODEL,
        "attribute": spec["attribute"],
        "comparison": spec["comparison"],
        "privileged_or_reference_group": priv,
        "unprivileged_or_comparison_group": "not " + str(priv),
        "n_included": int(mask.sum()),
        "positive_label_meaning": "in-hospital death / predicted high-risk flag",
    })
    fairness_rows.append(metrics)

    for group_value in sorted(prot.dropna().unique()):
        group_mask = prot == group_value
        rates = binary_confusion_rates(yt[group_mask], yh[group_mask])
        confusion_rows.append({
            "model": PRIMARY_MODEL,
            "attribute": spec["attribute"],
            "comparison": spec["comparison"],
            "group": group_value,
            "n": int(group_mask.sum()),
            "mortality_rate": float(yt[group_mask].mean()) if group_mask.sum() else np.nan,
            "predicted_positive_rate": float(yh[group_mask].mean()) if group_mask.sum() else np.nan,
            "mean_predicted_risk": float(yp[group_mask].mean()) if group_mask.sum() else np.nan,
            **rates,
        })
        observed_predicted_rows.append({
            "model": PRIMARY_MODEL,
            "attribute": spec["attribute"],
            "comparison": spec["comparison"],
            "group": group_value,
            "n": int(group_mask.sum()),
            "observed_mortality_rate": float(yt[group_mask].mean()) if group_mask.sum() else np.nan,
            "predicted_high_risk_rate": float(yh[group_mask].mean()) if group_mask.sum() else np.nan,
            "mean_predicted_risk": float(yp[group_mask].mean()) if group_mask.sum() else np.nan,
        })

fairness_metrics = pd.DataFrame(fairness_rows)
confusion_rates = pd.DataFrame(confusion_rows)
observed_vs_predicted = pd.DataFrame(observed_predicted_rows)

fairness_metrics.to_csv(RESULTS_DIR / "aif360_fairness_metrics_primary_model.csv", index=False)
confusion_rates.to_csv(RESULTS_DIR / "aif360_confusion_rates_primary_model.csv", index=False)
observed_vs_predicted.to_csv(RESULTS_DIR / "aif360_observed_vs_predicted_primary_model.csv", index=False)
display(fairness_metrics)

,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference,selection_rate_privileged,selection_rate_unprivileged,tpr_privileged,tpr_unprivileged,fpr_privileged,fpr_unprivileged,metric_backend,model,attribute,comparison,privileged_or_reference_group,unprivileged_or_comparison_group,n_included,positive_label_meaning
0,0.163355,1.825051,0.080400,0.106587,0.197994,0.361349,0.705314,0.785714,0.135744,0.268519,formula_equivalent_to_aif360_definitions,XGBoost,Age,>65 vs <=65,<=65,not <=65,4000,in-hospital death / predicted high-risk flag
1,0.035331,1.131634,0.008070,0.015451,0.268402,0.303733,0.753333,0.761404,0.192946,0.215779,formula_equivalent_to_aif360_definitions,XGBoost,Gender,Female vs Male,Male,not Male,3996,in-hospital death / predicted high-risk flag
2,0.246279,3.690595,0.114060,0.151112,0.091533,0.337812,0.652174,0.766234,0.060386,0.248550,formula_equivalent_to_aif360_definitions,XGBoost,ICUType,Other vs Cardiac Surgery Recovery Unit,Cardiac Surgery Recovery Unit,not Cardiac Surgery Recovery Unit,4000,in-hospital death / predicted high-risk flag
3,0.056645,1.240141,0.188606,0.102950,0.235880,0.292525,0.591549,0.780156,0.188324,0.205617,formula_equivalent_to_aif360_definitions,XGBoost,ICUType,Other vs Coronary Care Unit,Coronary Care Unit,not Coronary Care Unit,4000,in-hospital death / predicted high-risk flag
4,-0.216268,0.492175,-0.113753,-0.143886,0.425872,0.209604,0.814433,0.700680,0.321659,0.147639,formula_equivalent_to_aif360_definitions,XGBoost,ICUType,Other vs Medical ICU,Medical ICU,not Medical ICU,4000,in-hospital death / predicted high-risk flag
5,-0.002404,0.991585,0.000291,0.002335,0.285714,0.283310,0.757062,0.757353,0.199794,0.204173,formula_equivalent_to_aif360_definitions,XGBoost,ICUType,Other vs Surgical ICU,Surgical ICU,not Surgical ICU,4000,in-hospital death / predicted high-risk flag


In [18]:
# Threshold sensitivity analysis
thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
threshold_rows = []
for threshold in thresholds:
    y_pred_threshold = (y_prob_primary >= threshold).astype(int)
    for spec in fairness_specs:
        mask = spec["included_mask"]
        yt = pd.Series(y_test[mask]).reset_index(drop=True)
        yh = pd.Series(y_pred_threshold[mask]).reset_index(drop=True)
        prot = pd.Series(spec["protected_attribute"]).reset_index(drop=True)
        priv = spec["privileged_value"]
        metrics = compute_fairness_metrics(yt, yh, prot, priv)
        threshold_rows.append({
            "model": PRIMARY_MODEL,
            "threshold": threshold,
            "attribute": spec["attribute"],
            "comparison": spec["comparison"],
            "privileged_or_reference_group": priv,
            "statistical_parity_difference": metrics["statistical_parity_difference"],
            "disparate_impact_ratio": metrics["disparate_impact_ratio"],
            "equal_opportunity_difference": metrics["equal_opportunity_difference"],
            "average_odds_difference": metrics["average_odds_difference"],
        })
threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(RESULTS_DIR / "aif360_threshold_sensitivity_primary_model.csv", index=False)
display(threshold_sensitivity.head())

,model,threshold,attribute,comparison,privileged_or_reference_group,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference
0,XGBoost,0.1,Age,>65 vs <=65,<=65,0.124288,1.169719,0.009202,0.067173
1,XGBoost,0.1,Gender,Female vs Male,Male,0.074555,1.097482,0.002982,0.041528
2,XGBoost,0.1,ICUType,Other vs Cardiac Surgery Recovery Unit,Cardiac Surgery Recovery Unit,0.438097,1.962051,0.061507,0.252408
3,XGBoost,0.1,ICUType,Other vs Coronary Care Unit,Coronary Care Unit,-0.060138,0.929153,0.006302,-0.035889
4,XGBoost,0.1,ICUType,Other vs Medical ICU,Medical ICU,-0.205279,0.779841,-0.017007,-0.118221


In [19]:
# Save run metadata
metadata = {
    "primary_model": PRIMARY_MODEL,
    "aif360_available": AIF360_AVAILABLE,
    "aif360_import_error": None if AIF360_AVAILABLE else AIF360_IMPORT_ERROR,
    "positive_label": 1,
    "positive_label_meaning": "in-hospital death / predicted high-risk flag",
    "aif360_usage": "post-hoc fairness metric audit only; no AIF360 fairness-mitigation preprocessing was applied",
    "input_files": {
        "X_train": str(x_train_path),
        "X_test": str(x_test_path),
        "y_train": str(y_train_path),
        "y_test": str(y_test_path),
        "cohort_distribution_dataset": str(cohort_path),
    },
    "output_files": sorted(str(p) for p in RESULTS_DIR.glob("*.csv")),
}
with open(RESULTS_DIR / "aif360_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

{
  "primary_model": "XGBoost",
  "aif360_available": false,
  "aif360_import_error": "No module named 'aif360'",
  "positive_label": 1,
  "positive_label_meaning": "in-hospital death / predicted high-risk flag",
  "aif360_usage": "post-hoc fairness metric audit only; no AIF360 fairness-mitigation preprocessing was applied",
  "input_files": {
    "X_train": "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\data\\X_train.csv",
    "X_test": "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\data\\X_test.csv",
    "y_train": "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\data\\y_train.csv",
    "y_test": "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\data\\y_test.csv",
    "cohort_distribution_dataset": "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\data\\cohort_distribution_dataset.csv"
  },
  "output_files": [
    "c:\\Users\\junse\\Documents\\research\\IUBDC 2026\\AIF360\\results\\aif360\\aif360_confusion_rates_primary_model.c